In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SHEET_ID = "1jtBe-JnmljJnGY7xxFnzl-2K5Ujxn7Cpy8HV0HGOgO0" # ID da URL para acesso a planilha Google Sheets
GID = "0"  # aba Página1
url_tsv = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=tsv&gid={GID}"
print(url_tsv)
# TSV + vírgula decimal
df = pd.read_csv(url_tsv, sep="\t", decimal=",")
print("Colunas lidas:", list(df.columns))
display(df.head())



https://docs.google.com/spreadsheets/d/1jtBe-JnmljJnGY7xxFnzl-2K5Ujxn7Cpy8HV0HGOgO0/export?format=tsv&gid=0
Colunas lidas: ['acidez', 'teor', 'cor', 'categoria']


,acidez,teor,cor,categoria
0,7.4,13.2,4.5,Tinto
1,5.1,11.5,2.1,Branco
2,7.8,13.5,5.2,Tinto
3,4.9,12.1,2.8,Branco
4,8.1,14.2,6.0,Tinto


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SHEET_ID = "1jtBe-JnmljJnGY7xxFnzl-2K5Ujxn7Cpy8HV0HGOgO0" # ID da URL para acesso a planilha Google Sheets
GID = "0"  # aba Página1
url_tsv = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=tsv&gid={GID}"
print(url_tsv)
# TSV + vírgula decimal
df = pd.read_csv(url_tsv, sep="\t", decimal=",")
print("Colunas lidas:", list(df.columns))
display(df.head())

https://docs.google.com/spreadsheets/d/1jtBe-JnmljJnGY7xxFnzl-2K5Ujxn7Cpy8HV0HGOgO0/export?format=tsv&gid=0
Colunas lidas: ['acidez', 'teor', 'cor', 'categoria']


,acidez,teor,cor,categoria
0,7.4,13.2,4.5,Tinto
1,5.1,11.5,2.1,Branco
2,7.8,13.5,5.2,Tinto
3,4.9,12.1,2.8,Branco
4,8.1,14.2,6.0,Tinto


In [ ]:
FEATURE_SETS = {
    "BC":  ["acidez", "teor"],
    "BD":  ["acidez", "cor"],
    "CD":  ["teor", "cor"],
    "BCD": ["acidez", "teor", "cor"]
}
FEATURES = FEATURE_SETS["BCD"]   # <-- troque aqui: "BC", "BD", "CD", "BCD"
TARGET = "categoria"
for col in FEATURES + ["cor"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Normaliza o target e remove linhas inválidas
df[TARGET] = df[TARGET].astype(str).str.strip().str.lower()
df = df.dropna(subset=FEATURES + [TARGET]).copy()
print("\nDistribuição do target:")
print(df[TARGET].value_counts())

le = LabelEncoder()
y_all = le.fit_transform(df[TARGET])
X_all = df[FEATURES].values
print("\nClasses:", list(le.classes_))
print("Total linhas válidas:", len(df))

# Split FIXO: 5 teste + 23 treino
RANDOM_STATE = 42
df_shuffled = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_df  = df_shuffled.iloc[:5].copy()
train_df = df_shuffled.iloc[5:28].copy()  # 23 linhas (assumindo total=28)
X_train = train_df[FEATURES].values
y_train = le.transform(train_df[TARGET])
X_test  = test_df[FEATURES].values
y_test  = le.transform(test_df[TARGET])
print("\nTreino:", len(train_df), "| Teste:", len(test_df))
print("Cidades no TESTE:", test_df)

# Modelo KNN
K = 5
WEIGHTS = "distance"  # "uniform" ou "distance"
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=K, weights=WEIGHTS))
])
model.fit(X_train, y_train)

# Avaliação
y_pred = model.predict(X_test)
print("\nFeatures usadas:", FEATURES)
print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Matriz de confusão (real x previsto):")
print(confusion_matrix(y_test, y_pred))


Distribuição do target:
categoria
tinto     10
branco    10
Name: count, dtype: int64

Classes: ['branco', 'tinto']
Total linhas válidas: 20

Treino: 15 | Teste: 5
Cidades no TESTE:    acidez  teor  cor categoria
0     7.4  13.2  4.5     tinto
1     5.2  11.7  2.2    branco
2     5.6  12.2  2.7    branco
3     5.1  11.5  2.1    branco
4     8.5  14.5  6.2     tinto

Features usadas: ['acidez', 'teor', 'cor']
Acurácia: 1.0

Relatório:
              precision    recall  f1-score   support

      branco       1.00      1.00      1.00         3
       tinto       1.00      1.00      1.00         2

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5

Matriz de confusão (real x previsto):
[[3 0]
 [0 2]]
